# Campus Prediction Validation

Validates Random Forest allergenic tree species predictions against the University
of Twente campus tree inventory.

**Workflow**
1. Load and clean the campus inventory point dataset, retaining only the five
   target allergenic genera.
2. Load the predicted crown polygons (private trees only, no ground-truth labels).
3. Spatially join predictions with campus inventory points using the same
   species-assignment hierarchy as the ground-truth preprocessing step.
4. Clip to the campus extent and split into matched (inventory point inside crown)
   and unmatched (no inventory point) crowns.
5. Compute overall accuracy, per-species accuracy, and a confusion matrix for
   matched crowns.
6. Export the validated shapefile for further inspection in QGIS.

**Import libraries**

In [ ]:
import os
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from shapely.validation import make_valid

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.paths import EXTERNAL_DIR, PROCESSED_DIR

**Define paths**

In [ ]:
# Random Forest multi-class predictions from feature configuration 2 
RF_PREDICTIONS_DIR = PROCESSED_DIR / "rf_multiclass/enschede_rf_multiclass_predictions_fc2_v01.shp"

# University of Twente campus tree inventory
CAMPUS_INVENTORY_DIR = EXTERNAL_DIR / "ut_campus_gt/bomen_campus.shp"

**Load and clean campus inventory points**

In [ ]:
points_gdf = gpd.read_file(CAMPUS_INVENTORY_DIR)

# Retain only geometry and species name column
points_gdf = points_gdf[["geometry", "boomsoort"]]

# Keep only rows containing one of the five target allergenic genera.
# The regex matches the genus name anywhere in the species string,
# case-insensitively (e.g. "Quercus robur", "BETULA pendula").
regex_pattern = r"\b(QUERCUS|BETULA|ALNUS|FRAXINUS|PLATANUS)\b"
points_gdf = points_gdf[
    points_gdf["boomsoort"].str.contains(regex_pattern, case=False, na=False, regex=True)
]

# Standardise to genus name only (e.g. "Quercus robur" → "Quercus")
points_gdf["point_label"] = (
    points_gdf["boomsoort"]
    .str.split(" ")
    .str[0]
    .str.title()
)
points_gdf = points_gdf.drop(columns=["boomsoort"])

print(f"Campus inventory points (target species only): {len(points_gdf)}")
print(points_gdf["point_label"].value_counts())

**Load and clean predicted crown polygons**

In [ ]:
crowns_gdf = gpd.read_file(RF_PREDICTIONS_DIR)
crowns_gdf["id"] = crowns_gdf["id"].astype("int")

# Align CRS
if crowns_gdf.crs != points_gdf.crs:
    crowns_gdf = crowns_gdf.to_crs(points_gdf.crs)

# Repair invalid geometries and drop empty ones
crowns_gdf.geometry = crowns_gdf.geometry.apply(make_valid)
crowns_gdf = crowns_gdf[~crowns_gdf.geometry.is_empty]

# Remove exact duplicate polygons
crowns_gdf["geom_wkb"] = crowns_gdf.geometry.to_wkb()
crowns_gdf = crowns_gdf.drop_duplicates(subset="geom_wkb").reset_index(drop=True)
crowns_gdf = crowns_gdf.drop(columns=["geom_wkb"])

# Keep only private trees (is_allerge is NaN — no ground-truth public label)
crowns_gdf = crowns_gdf[crowns_gdf["is_allerge"].isna()].copy()
print(f"Private crown polygons: {len(crowns_gdf)}")

**Spatial join — match campus inventory points to crown polygons**

In [ ]:
crowns_with_points = gpd.sjoin(
    crowns_gdf,
    points_gdf[["point_label", "geometry"]],
    how="left",
    predicate="intersects",
)

# Attach point geometry 
crowns_with_points["point_geom"] = crowns_with_points["index_right"].apply(
    lambda idx: points_gdf.loc[idx, "geometry"] if pd.notnull(idx) else None
)

**Assign true species label to each crown**

In [ ]:
# Species-assignment hierarchy (same as crown_species_labelling.ipynb):
#   Case 0 — no inventory point inside crown → true_label = None
#   Case 1 — one point → assign its species directly
#   Case 2 — two points → species of the point closest to crown centroid
#   Case 3 — three or more points → majority vote

rows = []

for crown_id, group in crowns_with_points.groupby("id"):
    group = group.reset_index(drop=True)
    crown_geom   = group["geometry"].iloc[0]
    pred_species = group["full_speci"].iloc[0]

    has_points  = group["point_label"].notna()
    valid_pts   = group[has_points]

    if len(valid_pts) == 0:
        true_species = None
    elif len(valid_pts) == 1:
        true_species = valid_pts["point_label"].iloc[0]
    elif len(valid_pts) == 2:
        centroid  = crown_geom.centroid
        distances = gpd.GeoSeries(valid_pts["point_geom"]).distance(centroid)
        true_species = valid_pts.loc[distances.idxmin(), "point_label"]
    else:
        true_species = valid_pts["point_label"].mode().iloc[0]

    rows.append({
        "id":         int(crown_id),
        "pred_label": str(pred_species),
        "true_label": str(true_species) if true_species is not None else None,
        "geometry":   crown_geom,
    })

crowns_species_gdf = gpd.GeoDataFrame(rows, geometry="geometry", crs=crowns_with_points.crs)
print(f"Total campus crown polygons processed: {len(crowns_species_gdf)}")

**Clip to campus extent**

In [ ]:
# Restrict to the bounding box of the campus inventory to exclude crown polygons
# from outside the campus area that were included due to the spatial extent of
# the full private-tree prediction dataset.
minx, miny, maxx, maxy = points_gdf.total_bounds
crowns_species_gdf = crowns_species_gdf.cx[minx:maxx, miny:maxy]
print(f"Campus crowns after bounding-box clip: {len(crowns_species_gdf)}")

**Split into matched and unmatched crowns**

In [ ]:
# Matched: crown intersected with at least one campus inventory point
matched   = crowns_species_gdf[crowns_species_gdf["true_label"].notna()].copy()

# Unmatched: no inventory point inside crown — requires field validation
unmatched = crowns_species_gdf[crowns_species_gdf["true_label"].isna()].copy()

print(f"Total crown polygons on campus:     {len(crowns_species_gdf)}")
print(f"Matched to inventory points:        {len(matched)}")
print(f"Without inventory match (field val):{len(unmatched)}")

**Overall classification accuracy (matched crowns)**

In [ ]:
matched["correct"] = matched["pred_label"] == matched["true_label"]

n_correct   = matched["correct"].sum()
n_incorrect = (~matched["correct"]).sum()
accuracy    = n_correct / len(matched) * 100

print(f"Correctly predicted:   {n_correct} ({accuracy:.1f}%)")
print(f"Incorrectly predicted: {n_incorrect} ({100 - accuracy:.1f}%)")

**Per-species accuracy (matched crowns)**

In [ ]:
rows = []
for species in sorted(matched["true_label"].unique()):
    subset    = matched[matched["true_label"] == species]
    correct   = (subset["pred_label"] == species).sum()
    incorrect = len(subset) - correct
    rows.append({
        "Species":   species,
        "Total":     len(subset),
        "Correct":   correct,
        "Incorrect": incorrect,
        "Accuracy%": round(correct / len(subset) * 100, 1) if len(subset) > 0 else 0,
    })

per_species_df = pd.DataFrame(rows)
print(per_species_df.to_string(index=False))

**Confusion matrix (matched crowns)**

In [ ]:
confusion_matrix = pd.crosstab(
    matched["pred_label"],
    matched["true_label"],
    rownames=["Predicted"],
    colnames=["True"],
)

plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix — Campus Inventory Validation")
plt.xlabel("True")
plt.ylabel("Predicted")
plt.tight_layout()
plt.show()

**Export validated shapefile**

In [ ]:
out_path = PROCESSED_DIR / "tree_shp/enschede_rf_multiclass_validation_campus_v01.shp"
crowns_species_gdf.to_file(out_path)
print(f"Saved: {out_path}")